# GVHMR spike (throwaway)

Checks that GVHMR installs and runs on a free Colab GPU with a static camera, and records what it
outputs: shapes, confidence signals, joint angles, bone lengths, timings and peak GPU memory.

**Runtime → Change runtime type → T4 GPU** first.

**Privacy:** the video you upload goes to this Colab VM. The last step of the run cell deletes the
upload, the re-encoded copy and every GVHMR intermediate (which include image crops and features),
then lists what is left. Only `spike_result.json` (numbers, no images) is kept and downloaded.

**You need, from your own registrations** (not the third-party mirrors GVHMR's demo uses):
- `SMPLX_NEUTRAL.npz` from https://smpl-x.is.tue.mpg.de (Download → "SMPL-X v1.1", NPZ+PKL)
- `SMPL_NEUTRAL.pkl` from https://smpl.is.tue.mpg.de (Download → "SMPL for Python", the
  `basicmodel_neutral_lbs_10_207_0_v1.1.0.pkl` file; the upload cell renames it)

Each cell prints its wall time. If a cell fails, keep its output: that failure is a spike result too.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import sys; print(sys.version)

## 1. Install (~15 min)

GVHMR pins torch 2.3 + CUDA 12.1 and a pytorch3d wheel built for Python 3.10, which Colab no longer
runs, so it goes in its own Python 3.10 environment. `chumpy` and `cython_bbox` need build tools
present at install time, hence `--no-build-isolation`.

Each command stops the cell on failure (a plain `!` line carries on silently). Safe to rerun: it
skips the clone and the environment if they already exist.

In [ ]:
%%time
import os, subprocess

def sh(cmd):
    """Run a shell command, stream its output, and raise if it fails."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait():
        raise RuntimeError(f"exit {proc.returncode}: {cmd}")

os.chdir("/content")
sh("[ -d GVHMR ] || git clone -q https://github.com/zju3dv/GVHMR")
sh("cd GVHMR && git checkout -q ee960bb6e2ea2d381aa97f08e9b71ef320b624b1 && git log -1 --format=%H")
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
sh("command -v uv || curl -LsSf https://astral.sh/uv/install.sh | sh")
PY = "/content/gvhmr-env/bin/python"
# Colab has no 3.10; uv downloads a standalone build
sh(f"[ -x {PY} ] || (uv python install 3.10 && uv venv --seed --python 3.10 /content/gvhmr-env)")
sh(f"{PY} --version")
sh(f"{PY} -m pip --version || {PY} -m ensurepip --upgrade")  # an env made without --seed has no pip
sh(f"{PY} -m pip install -q numpy==1.23.5 cython 'setuptools>=68' wheel")
sh(f"{PY} -m pip install -q --no-build-isolation chumpy cython_bbox")
sh(f"cd /content/GVHMR && {PY} -m pip install -q -r requirements.txt && {PY} -m pip install -q -e .")
sh(f"{PY} -c \"import torch, pytorch3d, smplx; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'pytorch3d', pytorch3d.__version__)\"")

## 2. Weights

GVHMR, HMR2, ViTPose and YOLO checkpoints come from the authors' Google Drive folder (by
downloading you accept their licences). If `gdown` hits Drive's rate limit, download the folder in
a browser and upload it to `/content/GVHMR/inputs/checkpoints/`.

In [ ]:
%%time
%cd /content/GVHMR
sh(f"{PY} -m pip install -q gdown")
sh("mkdir -p inputs/checkpoints/body_models/smplx inputs/checkpoints/body_models/smpl")
sh("/content/gvhmr-env/bin/gdown -q --folder https://drive.google.com/drive/folders/1eebJ13FUEXrKBawHpJroW0sNSxLjh9xD -O inputs/checkpoints")
sh("find inputs/checkpoints -type f -exec ls -la {} \\;")

In [ ]:
# Upload the SMPL-X neutral .npz and the SMPL neutral .pkl; they are stored under the names GVHMR expects.
from google.colab import files
import shutil
for name in files.upload():
    dest = "smplx/SMPLX_NEUTRAL.npz" if name.endswith(".npz") else "smpl/SMPL_NEUTRAL.pkl"
    shutil.move(name, f"/content/GVHMR/inputs/checkpoints/body_models/{dest}")
!ls -la /content/GVHMR/inputs/checkpoints/body_models/*

## 3. Video

Upload the clip (the spike was planned around `videos/IMG_5970.mov`, the side-on right-arm curl
the `curl-side-*` fixtures come from). GVHMR treats every input as 30 fps, so it is resampled to a
constant 30 fps first, and scaled to 720 px wide (as the fixtures were) to fit in memory.

In [ ]:
import json, os, subprocess
from pathlib import Path
from google.colab import files
%cd /content
uploaded = list(files.upload())  # saved to the working directory, so cd first
SRC = os.path.abspath(uploaded[0])
probe = json.loads(subprocess.run(
    ["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
     "stream=width,height,r_frame_rate,avg_frame_rate,nb_frames,duration:stream_side_data=rotation",
     "-of", "json", SRC], capture_output=True, text=True).stdout)
print(probe)
Path("/content/probe.json").write_text(json.dumps(probe))
CLIP = "/content/GVHMR/inputs/spike_clip.mp4"
!ffmpeg -v error -y -i "{SRC}" -vf "fps=30,scale=720:-2" -an -c:v libx264 -crf 18 {CLIP}
!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,avg_frame_rate,nb_frames -of csv=p=0 {CLIP}

## 4. Run GVHMR and dump what it outputs

The run, the summary and the clean-up are one cell, so the video is deleted even if the run fails.

In [ ]:
%%writefile /content/GVHMR/spike_run.py
import json, sys, time
from pathlib import Path

import numpy as np
import torch

CLIP, SOURCE_PROBE = sys.argv[1], json.loads(Path(sys.argv[2]).read_text())
sys.argv = ["demo.py", "--video", CLIP, "-s", "--output_root", "outputs/spike"]
sys.path.insert(0, "tools/demo")
import demo  # GVHMR's own demo script, for its preprocessing and config
import hydra
from hmr4d.utils.net_utils import detach_to_cpu

COCO17 = ["nose", "l_eye", "r_eye", "l_ear", "r_ear", "l_shoulder", "r_shoulder", "l_elbow", "r_elbow",
          "l_wrist", "r_wrist", "l_hip", "r_hip", "l_knee", "r_knee", "l_ankle", "r_ankle"]
# First 22 SMPL-X joints, as returned by EnDecoder.fk_v2.
J = dict(pelvis=0, l_hip=1, r_hip=2, spine1=3, l_knee=4, r_knee=5, spine2=6, l_ankle=7, r_ankle=8, spine3=9,
         l_foot=10, r_foot=11, neck=12, l_collar=13, r_collar=14, head=15, l_shoulder=16, r_shoulder=17,
         l_elbow=18, r_elbow=19, l_wrist=20, r_wrist=21)
STATIC_JOINTS = ["l_ankle", "l_foot", "r_ankle", "r_foot", "l_wrist", "r_wrist"]  # postprocess.pp_static_joint


def shapes(obj):
    if torch.is_tensor(obj):
        return {"shape": list(obj.shape), "dtype": str(obj.dtype)}
    if isinstance(obj, dict):
        return {k: shapes(v) for k, v in obj.items()}
    return type(obj).__name__


def angle(a, b, c):
    u, v = a - b, c - b
    cos = (u * v).sum(-1) / (np.linalg.norm(u, axis=-1) * np.linalg.norm(v, axis=-1) + 1e-9)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))


def stats(x):
    x = np.asarray(x, dtype=float)
    return {"min": float(x.min()), "p10": float(np.percentile(x, 10)), "median": float(np.median(x)),
            "p90": float(np.percentile(x, 90)), "max": float(x.max())}


result = {"source_probe": SOURCE_PROBE, "torch": torch.__version__, "gpu": torch.cuda.get_device_name(), "timings_s": {}}
t0 = time.time()
cfg = demo.parse_args_to_cfg()
result["timings_s"]["copy"] = time.time() - t0

t0 = time.time()
demo.run_preprocess(cfg)
result["timings_s"]["preprocess_yolo_vitpose_vitfeatures"] = time.time() - t0
data = demo.load_data_dict(cfg)
result["frames"] = int(data["length"])

t0 = time.time()
model = hydra.utils.instantiate(cfg.model, _recursive_=False)
model.load_pretrained_model(cfg.ckpt_path)
model = model.eval().cuda()
result["timings_s"]["load_model"] = time.time() - t0
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
with torch.no_grad():
    pred = model.predict(data, static_cam=True)
torch.cuda.synchronize()
result["timings_s"]["predict"] = time.time() - t0
result["peak_gpu_mb_predict"] = torch.cuda.max_memory_allocated() / 2**20
pred = detach_to_cpu(pred)
result["output_shapes"] = shapes(pred)

# Joints in GVHMR's gravity-aligned world frame, and in the camera frame.
endecoder = model.pipeline.endecoder
with torch.no_grad():
    batch = lambda params: {k: v[None].cuda() for k, v in params.items() if k in ("body_pose", "betas", "global_orient", "transl")}
    world = endecoder.fk_v2(**batch(pred["smpl_params_global"]))[0].cpu().numpy()  # (L, 22, 3)
    incam = endecoder.fk_v2(**batch(pred["smpl_params_incam"]))[0].cpu().numpy()
    skeleton = endecoder.smplx_model.get_skeleton(pred["smpl_params_global"]["betas"][None].cuda())[0, :, :22].cpu().numpy()

# Which axis is up: the one along which the shoulders sit furthest above the hips.
torso = world[:, [J["l_shoulder"], J["r_shoulder"]]].mean(1) - world[:, [J["l_hip"], J["r_hip"]]].mean(1)
mean_torso = torso.mean(0)
result["world_torso_vector_mean"] = mean_torso.tolist()
up = np.zeros(3); up[np.abs(mean_torso).argmax()] = np.sign(mean_torso[np.abs(mean_torso).argmax()])
result["world_up_axis_inferred"] = up.tolist()
floor_axis_min = world[..., np.abs(up).argmax()].min(1)
result["lowest_joint_along_up_stats"] = stats(floor_axis_min)

j = lambda arr, name: arr[:, J[name]]
angles = {}
for s in ("l", "r"):
    angles[f"{s}_elbow"] = angle(j(world, f"{s}_shoulder"), j(world, f"{s}_elbow"), j(world, f"{s}_wrist"))
    angles[f"{s}_shoulder"] = angle(j(world, f"{s}_hip"), j(world, f"{s}_shoulder"), j(world, f"{s}_elbow"))
    angles[f"{s}_hip"] = angle(j(world, f"{s}_shoulder"), j(world, f"{s}_hip"), j(world, f"{s}_knee"))
    angles[f"{s}_knee"] = angle(j(world, f"{s}_hip"), j(world, f"{s}_knee"), j(world, f"{s}_ankle"))
    angles[f"{s}_ankle"] = angle(j(world, f"{s}_knee"), j(world, f"{s}_ankle"), j(world, f"{s}_foot"))
cos_lean = (torso @ up) / np.linalg.norm(torso, axis=1)
angles["trunk_lean"] = np.degrees(np.arccos(np.clip(cos_lean, -1, 1)))
# Interior angles should not depend on the frame; check by recomputing one in camera coordinates.
incam_elbow = angle(j(incam, "r_shoulder"), j(incam, "r_elbow"), j(incam, "r_wrist"))
result["r_elbow_world_vs_incam_max_abs_diff"] = float(np.abs(incam_elbow - angles["r_elbow"]).max())
result["angle_stats"] = {k: stats(v) for k, v in angles.items()}

# Bone lengths: from the posed joints per frame (does shape drift over the clip?) and the rest skeleton.
bones = {"upper_arm": ("shoulder", "elbow"), "forearm": ("elbow", "wrist"), "thigh": ("hip", "knee"), "shin": ("knee", "ankle")}
result["bone_lengths_m"] = {}
for s in ("l", "r"):
    for bone, (a, b) in bones.items():
        per_frame = np.linalg.norm(j(world, f"{s}_{a}") - j(world, f"{s}_{b}"), axis=1)
        rest = np.linalg.norm(skeleton[:, J[f"{s}_{a}"]] - skeleton[:, J[f"{s}_{b}"]], axis=1)
        result["bone_lengths_m"][f"{s}_{bone}"] = {"posed": stats(per_frame), "rest": stats(rest)}
betas = pred["smpl_params_global"]["betas"].numpy()  # (L, 10): one estimate per frame
result["betas_mean"] = betas.mean(0).tolist()
result["betas_per_frame_std"] = betas.std(0).tolist()
result["betas_per_frame_std_max"] = float(betas.std(0).max())
result["betas_first_vs_last_quarter_max_abs_diff"] = float(np.abs(betas[: len(betas) // 4].mean(0) - betas[-(len(betas) // 4):].mean(0)).max())
result["rest_height_m_approx"] = float(np.ptp(skeleton[0, :, 1]))  # joint span only, below true height

# Confidence signals that exist: ViTPose 2D keypoint scores (GVHMR's input) and the foot/wrist
# "static" (contact) logits from the network.
vitpose = data["kp2d"].numpy()  # (L, 17, 3): x, y, heatmap score
result["vitpose_conf"] = {name: stats(vitpose[:, i, 2]) for i, name in enumerate(COCO17)}
static_logits = pred["net_outputs"].get("static_conf_logits")
static = torch.sigmoid(static_logits[0]).cpu().numpy() if static_logits is not None else None
result["static_conf"] = None if static is None else {name: stats(static[:, i]) for i, name in enumerate(STATIC_JOINTS)}
result["net_outputs_keys"] = sorted(pred["net_outputs"].keys())

# Per-frame series, for comparing with the MediaPipe fixture recorded from the same clip.
result["series"] = [
    {"t_ms": round(i * 1000 / 30, 1),
     **{k: round(float(v[i]), 1) for k, v in angles.items()},
     "vitpose_conf": {n: round(float(vitpose[i, k, 2]), 3) for k, n in enumerate(COCO17) if k >= 5},
     **({"static": [round(float(x), 3) for x in static[i]]} if static is not None else {})}
    for i in range(result["frames"])
]
Path("/content/spike_result.json").write_text(json.dumps(result, indent=1))
print(json.dumps({k: v for k, v in result.items() if k != "series"}, indent=1))

In [ ]:
%%time
import json, shutil, glob, os
try:
    !cd /content/GVHMR && /content/gvhmr-env/bin/python spike_run.py {CLIP} /content/probe.json
finally:
    for path in [SRC, CLIP, "/content/probe.json", "/content/GVHMR/outputs"]:
        if os.path.isdir(path):
            shutil.rmtree(path, ignore_errors=True)
        elif os.path.exists(path):
            os.remove(path)
    uploaded = SRC = None
    # Everything on the VM that is not the install (env, GVHMR's tracked source, model weights) or our result.
    !cd /content/GVHMR && git status --porcelain --untracked-files=all --ignored | grep -v -e inputs/checkpoints/ -e '\.egg-info/' -e __pycache__ -e spike_run.py
    !find /content -mindepth 1 -maxdepth 1 -not -name GVHMR -not -name gvhmr-env -not -name sample_data -not -name spike_result.json -not -name '.config'
    print("Video files left on the VM:", [p for root in ("/content", "/tmp", "/root") for ext in ("mp4", "mov", "MOV", "MP4", "m4v")
                                          for p in glob.glob(f"{root}/**/*.{ext}", recursive=True)
                                          if not p.startswith(("/content/gvhmr-env/", "/content/GVHMR/docs/"))] or "none")

In [ ]:
from google.colab import files
files.download("/content/spike_result.json")